# 1. Data Analysis and Problem Definition

The objective of this notebook is to perform an initial inspection of the raw dataset. We aim to understand the data structure, identify data quality issues, analyze the distribution of the target variable to detect outliers and formally define the machine learning problem we aim to solve.

**Steps:**
1.  **Data Structure Inspection:** Load the dataset to understand the available features and data types.
2.  **Quality Assessment:** Identify missing values and potential data quality issues.
3.  **Target Analysis:** Analyze the distribution of the target variable to detect outliers and signs of heteroscedasticity.
4.  **Problem Definition:** Formally define the machine learning problem based on the exploratory findings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

## Dataset Overview

In [ ]:
pd.set_option('display.max_columns', None)
df = pd.read_csv('../../data/madrid/listings.csv')
display(df.info())

## Head Inspection

In [ ]:
df.head()

## Missing Values Analysis

In [ ]:
# Calculate missing values percentage
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_info = pd.DataFrame({
    'Missing Values': missing_count,
    'Percentage': missing_percentage
})

# Display columns with missing values sorted
missing_info = missing_info[missing_info['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)
display(missing_info)

## Descriptive Statistics

In [ ]:
df.describe(include=np.number)

## Target Variable Analysis: Price

In [ ]:
# Clean characters ('$') and convert to float if necessary
if df['price'].dtype == 'object':
    df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)

# Price statistics with specific percentiles
display(df['price'].describe(percentiles=[0.25, 0.275, 0.30, 0.325, 0.35, 0.375, 0.4, 0.425, 0.45, 0.475, 0.5, 0.525, 0.55, 0.575, 0.6, 0.625, 0.65, 0.675, 0.7, 0.725, 0.75, 0.775, 0.8, 0.825, 0.85, 0.875, 0.90, 0.925, 0.95, 0.975, 0.99]))

## Outliers Visualization

In [ ]:
save_dir = '../analysis'
os.makedirs(save_dir, exist_ok=True) 

# Full Price Distribution
plt.figure(figsize=(12, 6))
sns.histplot(df['price'], bins=100, kde=True, color='steelblue')
plt.title('Price Distribution')
plt.xlabel('Price (€)')
plt.ylabel('Frequency')
plt.savefig(f'{save_dir}/price_full_distribution.png') 
plt.show()

# Boxplot for Outlier Detection
plt.figure(figsize=(12, 6))
sns.boxplot(x=df['price'], color='orange')
plt.title('Price Boxplot')
plt.xlabel('Price (€)')
plt.savefig(f'{save_dir}/price_boxplot.png')
plt.show()

## Common Market Visualization

In [ ]:
# Price Distribution (€0 - €249)
plt.figure(figsize=(12, 6))
sns.histplot(df[df['price'] <= 249]['price'], bins=50, kde=True, color='steelblue')
plt.title('Price Distribution (€0 - €249)')
plt.xlabel('Price (€)')
plt.ylabel('Frequency')
plt.savefig(f'{save_dir}/price_0_249_distribution.png')
plt.show()

## Post-Cleaning Missing Values Analysis

In the upcoming preprocessing pipeline, rows with missing values in the target variable (`price`) will be dropped immediately, as they cannot be used for supervised training.

The following analysis simulates this initial cleaning step to identify the remaining missing values in the dataset. This insight is crucial for defining the specific imputation strategies required for the predictor variables in the final pipeline.

In [ ]:
# Drop rows where target is missing
df_price_filtered = df.dropna(subset=['price'])

missing_count = df_price_filtered.isnull().sum()
missing_percentage = (df_price_filtered.isnull().sum() / len(df_price_filtered)) * 100

missing_info = pd.DataFrame({
    'Missing Values': missing_count,
    'Percentage': missing_percentage
})

missing_info = missing_info[missing_info['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)
display(missing_info)

## Problem Definition

Based on the exploratory data analysis, we define the scope and parameters for our machine learning problem:

* **Goal:** To build a robust regression model that predicts the daily listing price. Crucially, the model must quantify the uncertainty of its predictions, providing confidence intervals rather than just point estimates.
* **Problem Type:** Supervised Regression.
* **Target Variable:** `price`. We observed significant right-skewness and outliers; therefore, we will focus the model on the main market segment (e.g., properties under €249, covering the vast majority of the distribution).
* **Input Features:** We have selected a comprehensive set of predictors categorized by domain (only some of the columns are specified):
    * **Location & Geography:**
        * Exact coordinates: `latitude`, `longitude`.
        * Administrative zones: `neighbourhood_cleansed` (granular), `neighbourhood_group_cleansed` (broad).
    * **Property Structure & Capacity:**
        * Size metrics: `accommodates`, `bedrooms`, `beds`, `bathrooms`.
        * Typology: `room_type`, `property_type`.
        * Features: `amenities_count`.
    * **Host Profile & Management:**
        * Professionalism: `calculated_host_listings_count`, `host_response_time`.
        * Trust factors: `host_is_superhost`, `host_identity_verified`.
        * Booking rules: `minimum_nights_avg_ntm`, `instant_bookable`.
        * Time-derived metrics: `host_days_active`, `days_since_first_review`, `days_since_last_review`.
    * **Reputation & Demand:**
        * Volume: `number_of_reviews`, `reviews_per_month`.
        * Quality (Scores): `review_scores_rating` and specific sub-scores (cleanliness, location, value, etc.).
        * Availability: `availability_30`, `availability_365` (proxies for current demand).